### Output Parsers in LangChain

This section introduces the most important pattern in LangChain: a prompt -> model -> parser pipeline.

Why it matters:
- LLMs usually return raw text, not structured data.
- Output parsers convert the model response into Python types you can use in code.
- This helps when building apps, APIs, or agent workflows that need valid JSON, objects, or plain strings.

Core idea:
- `PromptTemplate` creates the input prompt.
- `ChatOpenAI` sends the prompt to the model.
- `StrOutputParser`, `JsonOutputParser`, or `PydanticOutputParser` transforms the model result into a usable format.
- The `|` operator composes the chain, so each step passes its output to the next stage.

A good AI engineer should remember:
- Use `StrOutputParser` for text generation.
- Use `JsonOutputParser` for structured JSON but be careful about schema control.
- Use `PydanticOutputParser` when strict validation and type safety are important.


#### 1. StrOutputParser

`StrOutputParser` is the simplest parser.
It turns the model output into a plain Python string.

Use it when:
- you want natural language text
- you are summarizing or generating essays
- you do not need strict structure



In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
model = ChatOpenAI()

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

#### 2. Json OutputParser

`JsonOutputParser` is used when you want the model to produce valid JSON.

Key idea:
- The model is asked to return content in JSON format.
- The parser converts the raw JSON text into a Python dictionary.

However, there is an important AI-engineering point:
- JSON parsers do not guarantee specific keys unless you constrain the schema.
- The model may return `fact1`, `fact2`, `fact3`, etc., instead of a standardized list under `facts`.

This is because the model is generating valid JSON, but the exact field names are still open-ended unless you enforce a schema.

Common use cases:
- extracting structured metadata
- returning dictionaries for API integrations
- working with tools that expect JSON objects

Best practice:
- If exact output shape matters, use a schema-based parser such as `PydanticOutputParser` or a strict JSON schema.


In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Define the model
model = ChatOpenAI()

parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

{'fact1': 'Black holes are regions in space where gravitational forces are so strong that nothing, not even light, can escape.', 'fact2': 'The center of a black hole is known as the singularity, where gravity becomes infinite and spacetime is infinitely curved.', 'fact3': 'Black holes can be formed when massive stars collapse under their own gravity, or through the merging of two or more black holes.', 'fact4': 'Black holes can vary in size, with supermassive black holes found at the centers of galaxies and smaller stellar black holes formed from dying stars.', 'fact5': 'Black holes are theorized to emit radiation known as Hawking radiation, named after physicist Stephen Hawking who proposed its existence.'}


#### 3. Pydantic OutputParser

`PydanticOutputParser` is the most structured and reliable option when you need validation.

Why it matters for AI engineering:
- LLM outputs can be inconsistent.
- Pydantic gives a strict schema for the output.
- The model response is parsed into a Python class instance with validated fields.

In this notebook, the model is asked to generate a fictional person's name, age, and city.

```python
class Person(BaseModel):
    name: str
    age: int = Field(gt=18)
    city: str
```

This means:
- `name` must be a string
- `age` must be an integer greater than 18
- `city` must be a string

If the model output violates the schema, the parser can reject or help guide the model to a valid output.

This is an important pattern for production AI systems because you want predictable, validated data for downstream code, APIs, and UIs.

Practical takeaway:
- Use strings for free-form text
- Use JSON for semi-structured data
- Use Pydantic for strict structured outputs


In [5]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

load_dotenv()

# Define the model
load_dotenv()

# Define the model
model = ChatOpenAI()

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)


name='Samantha' age=34 city='Colombo'
